<h2>MapReduce Mini-Project: Analyzing Amazon Movie Reviews</h2>

<p>
In this exercise, you will work as a data engineer for a streaming platform.
Your goal is to perform several analytics tasks on a free and publicly
available dataset of Amazon Movie Reviews using MapReduce in Hadoop.
</p>

<p>
You will complete four tasks:
</p>

<ol>
  <li><b>Count total number of reviews per movie</b></li>
  <li><b>Compute average rating per movie</b></li>
  <li><b>Extract frequent keywords from reviews</b></li>
  <li><b>Join average ratings with top keywords</b></li>
</ol>

<p>
For each task, you will write a MapReduce program (Python Streaming or Java)
and run it using Hadoop in local mode. Your final outputs will help the
company understand which movies are popular, how viewers rate them, and what
keywords often appear in the reviews.
</p>

<h2>About the Dataset</h2>

<p>
We will use the <b>Amazon Movies &amp; TV 5-core dataset</b>, which is publicly
available and contains movie reviews from Amazon. Each entry in the dataset
is stored as a JSON object with fields such as:
</p>

<ul>
  <li><code>reviewerID</code> – the ID of the reviewer</li>
  <li><code>asin</code> – unique movie identifier</li>
  <li><code>reviewText</code> – full written review</li>
  <li><code>overall</code> – the star rating (1 to 5)</li>
  <li><code>vote</code> – how many users found the review helpful</li>
  <li><code>category</code> – always “Movies &amp; TV” in this dataset</li>
</ul>

<p>
You will download the dataset and inspect a few records to understand its
structure before starting the tasks.
</p>

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz

In [ ]:

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += f":{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin"

In [ ]:
import gzip
import json
import os
import sys # Import sys for printing warnings to stderr

# -------------------------------------------------------------------
# 1) Download the SMALL Movies & TV dataset (correct version)
# -------------------------------------------------------------------
print("Downloading SMALL Movies & TV 5-core dataset...")

URL = "https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Movies_and_TV_5.json.gz"
FILE_GZ = "Movies_and_TV_small.json.gz"

!wget --no-check-certificate -O {FILE_GZ} {URL}

if os.path.getsize(FILE_GZ) == 0:
    raise ValueError("Downloaded file is empty!")

print("Download complete.\n")

# -------------------------------------------------------------------
# 2) Load JSON data (each line is a JSON object)
# -------------------------------------------------------------------
print("Loading JSON data from JSON Lines format...")

data = []
with gzip.open(FILE_GZ, "rt", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if line: # Only process non-empty lines
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Could not decode JSON on line {line_num}: {line}. Error: {e}", file=sys.stderr)
                # Continue to the next line to be robust against malformed lines
                continue

print(f"Total records loaded: {len(data)}") # Should be ~3.4 million records
print()

# -------------------------------------------------------------------
# 3) Convert to JSON-LINES format for MapReduce (if not already done)
#    This step ensures 'movies.json' is a clean JSON-Lines file.
# -------------------------------------------------------------------
print("Converting to JSON-lines format (outputting to movies.json with 900,000 records)...")

# Limit to 900,000 records to have less runing times on colab (on a real cluster, remove this line)
limited_data = data[:900000]

with open("movies.json", "w", encoding="utf-8") as out:
    for entry in limited_data:
        out.write(json.dumps(entry) + "\n")

print(f"Conversion complete. Saved as movies.json with {len(limited_data)} records\n")

# -------------------------------------------------------------------
# 4) Preview
# -------------------------------------------------------------------
print("Sample entries:\n")

with open("movies.json", "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        if not line: # Check for end of file
            print("Not enough lines in movies.json to display 3 samples.")
            break
        print(json.loads(line))

--2025-12-10 10:05:33--  https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Movies_and_TV_5.json.gz
Resolving jmcauley.ucsd.edu (jmcauley.ucsd.edu)... 137.110.160.73
Connecting to jmcauley.ucsd.edu (jmcauley.ucsd.edu)|137.110.160.73|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 791322468 (755M) [application/x-gzip]
Saving to: ‘Movies_and_TV_small.json.gz’

Movies_and_TV_small 100%[===================>] 754.66M  55.5MB/s    in 13s     

2025-12-10 10:05:47 (56.0 MB/s) - ‘Movies_and_TV_small.json.gz’ saved [791322468/791322468]

Download complete.

Loading JSON data from JSON Lines format...
Total records loaded: 3410019

Converting to JSON-lines format (outputting to movies.json with 900,000 records)...
Conversion complete. Saved as movies.json with 900000 records

Sample entries:

{'overall': 5.0, 'verified': True, 'reviewTime': '11 9, 2012', 'reviewerID': 'A2M1CU2IRZG0K9', 'asin': '0005089549', 'st

<h2>Task 1 — Count Total Number of Reviews per Movie</h2>

<p>
Your first task is to count how many reviews each movie has received. You will
write a MapReduce program where:
</p>

<ul>
  <li>The <b>mapper</b> reads each JSON record, extracts the <code>asin</code>
      field, and emits <code>(asin, 1)</code>.</li>
  <li>The <b>reducer</b> sums the counts for each movie and outputs
      <code>(asin, total_reviews)</code>.</li>
</ul>

<p>
This task is conceptually similar to a word count, but applied to movie IDs.
Complete the mapper and reducer code in the following cell.
</p>

In [ ]:
%%writefile mapper_task1.py
import sys
import json

def mapper():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            review = json.loads(line)
            asin = review.get('asin')
            if asin:
                print(f"{asin}\t1")
        except json.JSONDecodeError:
            # Skip malformed JSON lines
            continue

if __name__ == '__main__':
  mapper()

Writing mapper_task1.py


In [ ]:
# Affiche les premières émissions avec le séparateur visible
!python mapper_task1.py < movies.json | head -5

{"overall": 5.0, "verified": true, "reviewTime": "11 9, 2012", "reviewerID": "A2M1CU2IRZG0K9", "asin": "0005089549", "style": {"Format:": " VHS Tape"}, "reviewerName": "Terri", "reviewText": "So sorry I didn't purchase this years ago when it first came out!!  This is very good and entertaining!  We absolutely loved it and anticipate seeing it repeatedly.  We actually wore out the cassette years back, so we also purchased this same product on cd.  Best purchase we made out of all!  Would purchase on dvd if we could find one.", "summary": "Amazing!", "unixReviewTime": 1352419200}
0005089549	1
{"overall": 5.0, "verified": true, "reviewTime": "12 30, 2011", "reviewerID": "AFTUJYISOFHY6", "asin": "0005089549", "style": {"Format:": " VHS Tape"}, "reviewerName": "Melissa D. Abercrombie", "reviewText": "Believe me when I tell you that you will receive a blessing watching this video of the Cathedral Quartet.  They bring back most of the singers that were ever in their group and it is a really g

In [ ]:
%%writefile reduce_task1.py
import sys

def reducer():
    current_asin = None
    current_count = 0

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        try:
            asin, count_str = line.split('\t', 1)
            count = int(count_str)
        except ValueError:
            continue

        if current_asin == asin:
            current_count += count
        else:
            if current_asin:
                # Output result for the previous asin
                print(f"{current_asin}\t{current_count}")
            current_asin = asin
            current_count = count

    # Output the last asin's count
    if current_asin:
        print(f"{current_asin}\t{current_count}")

if __name__ == '__main__':
  reducer()

Writing reduce_task1.py


In [ ]:
!python reduce_task1.py < movies.json

In [ ]:
# Just on a loop sample
!head -1000 movies.json | python mapper_task1.py | sort | python reduce_task1.py | head -5

print("Top 10 famous movies:")
!python mapper_task1.py < movies.json | sort | python reduce_task1.py > task1_output.txt
!sort -k2 -nr task1_output.txt | head -10

0005019281	300
000503860X	5
0005089549	2
0005092663	14
0005119367	220
Top 10 famous movies:
B00006CXSS	5708
6305837325	3682
0790729628	2365
B0001VL0KC	2339
B00005JLF2	2168
0793906091	2048
7799133104	2008
B00003CWT6	1912
6304994567	1858
6304994540	1858


<h2>Task 2 — Compute Average Rating per Movie</h2>

<p>
In this task, you will compute the <b>average rating</b> for each movie.
</p>

<p>The mapper should:</p>
<ul>
  <li>Extract <code>asin</code> and <code>overall</code> (rating)</li>
  <li>Emit <code>(asin, rating)</code></li>
</ul>

<p>The reducer should:</p>
<ul>
  <li>Sum all ratings for each movie</li>
  <li>Count how many ratings were received</li>
  <li>Compute and output the average rating</li>
</ul>

<p>
Use a MapReduce job to generate a list of movies with their average ratings.
</p>

In [ ]:
%%writefile mapper_task2.py
import sys
import json

def mapper():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            review = json.loads(line)
            asin = review.get('asin')
            overall_rating = review.get('overall')
            if asin is not None and overall_rating is not None:
                print(f"{asin}\t{overall_rating}")
        except json.JSONDecodeError:
            continue

if __name__ == '__main__':
  mapper()

Writing mapper_task2.py


In [ ]:
%%writefile reducer_task2.py
import sys

def reducer():
    current_asin = None
    sum_ratings = 0.0
    count_ratings = 0

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            asin, rating_str = line.split('\t', 1)
            rating = float(rating_str)
        except ValueError:
            continue

        if current_asin == asin:#Find and accumulate value
            sum_ratings += rating
            count_ratings += 1
        else:
            if current_asin  and count_ratings > 0:
                average_rating = sum_ratings / count_ratings
                print(f"{current_asin}\t{average_rating}")

            # update value
            current_asin = asin
            sum_ratings = rating
            count_ratings = 1

    # For the last line
    if current_asin is not None and count_ratings > 0:
        average_rating = sum_ratings / count_ratings
        print(f"{current_asin}\t{average_rating}")

if __name__ == '__main__':
    reducer()

Writing reducer_task2.py


Now, let's test the mapper and reducer for Task 2 on a sample of the data and save the output to `task2_output.txt`.

In [ ]:
!head -1000 movies.json | python mapper_task2.py | sort | python reducer_task2.py | head -5

!head -1000 movies.json | python mapper_task2.py | sort | python reducer_task2.py > task2_output.txt

print("Top of Ten movies with the highest average ratings:")
!sort -k2 -nr task2_output.txt | head -10

0005019281	4.42
000503860X	5.0
0005089549	5.0
0005092663	4.571428571428571
0005119367	4.781818181818182
Top of Ten movies with the highest average ratings:
0740318764	5.0
0578047861	5.0
0005089549	5.0
000503860X	5.0
0307142493	4.81203007518797
0005119367	4.781818181818182
0757915655	4.75
0310271908	4.75
0615115187	4.681818181818182
0764005529	4.655172413793103


<h2>Task 3 — Extract Frequent Keywords from Reviews</h2>

<p>
Now you will perform text analysis on the <code>reviewText</code> field.
Your task is to extract meaningful keywords for each movie.
</p>

<p>The mapper should:</p>
<ul>
  <li>Clean and tokenize the text</li>
  <li>Remove punctuation and stopwords</li>
  <li>Emit <code>(asin:word, 1)</code> for each keyword</li>
</ul>

<p>The reducer should:</p>
<ul>
  <li>Sum the counts for each <code>(asin, word)</code> pair</li>
  <li>Output the total frequency of each keyword per movie</li>
</ul>

<p>
This task combines text preprocessing with distributed computation.
</p>

In [ ]:
%%writefile mapper_task3.py
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import json
import sys
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
# Write your Mapper and Reducer code for Task 3 here.
def clean_text(text):
    tokens = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    filtered = [word for word in tokens if word.isalpha() and word not in stop_words]
    return filtered
def mapper():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            review = json.loads(line)
            asin = review.get('asin')
            overall_rating = review.get('overall')
            reviewtext = review.get('reviewText')
            reviewtext_token=clean_text(reviewtext)
            if asin and reviewtext_token is not None:
                for token in reviewtext_token:
                  print(f"{asin}:{token}\t1")
        except json.JSONDecodeError:
            # Skip malformed JSON lines
            continue


if __name__ == '__main__':
  mapper()


Overwriting mapper_task3.py


In [ ]:
%%writefile reduce_task3.py
import sys

def reducer():
    current_key = None
    current_count = 0

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        try:
            key, count_str = line.split('\t', 1)
            count = int(count_str)
        except ValueError:
            continue

        if current_key == key:
            current_count += count
        else:
            if current_key:
                # Output result for the previous key
                print(f"{current_key}\t{current_count}")
            current_key = key
            current_count = count

    # Output the last key's count
    if current_key:
        print(f"{current_key}\t{current_count}")

if __name__ == '__main__':
  reducer()

Overwriting reduce_task3.py


In [ ]:
print("Running MapReduce job for Task 3")

!head -1000 movies.json |python mapper_task3.py| sort | python reduce_task3.py > task3_output.txt

print("\nFirst 10 lines of task3_output.txt:")
!head -10 task3_output.txt

Running MapReduce job for Task 3
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.

First 10 lines of task3_output.txt:
0005019281:abandoned	2
0005019281:abc	4
0005019281:ability	3
0005019281:able	2
0005019281:accurate	1
0005019281:across	2
0005019281:act	1
0005019281:acted	3
0005019281:acting	22
0005019281:actor	15


<h2>Task 4 — Join Ratings with Top Keywords</h2>

<p>
For this task, you will combine the results of Task 2 (average ratings) and
Task 3 (keyword frequencies) using a <b>reduce-side join</b>.
</p>

<p>
You will provide two inputs to your MapReduce job:
</p>

<ul>
  <li><b>Ratings file</b> with <code>(asin, average_rating)</code></li>
  <li><b>Keywords file</b> with <code>(asin, keyword, count)</code></li>
</ul>

<p>Each mapper should tag its data:</p>

<ul>
  <li><code>("R", rating)</code> for ratings</li>
  <li><code>("K", keyword:count)</code> for keywords</li>
</ul>

<p>
The reducer will receive all entries for a given movie and combine them to
produce an output containing:
</p>

<ul>
  <li>The movie identifier (<code>asin</code>)</li>
  <li>Its average rating</li>
  <li>Its most frequent keywords</li>
</ul>

In [ ]:
# Write your Mapper and Reducer code for Task 4 here.
%%writefile mapper_task4.py
import sys
def mapper():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        try:
            if '\t' not in line:
                continue

            key_part, value = line.split('\t', 1)

            # sure to be in the right cluster
            if ':' in key_part:
                # Task 3: format
                asin, keyword = key_part.split(':', 1)
                print(f"{asin}\tK:{keyword}:{value}")
            else:
                # Task 2: format for rating
                asin = key_part
                rating = value
                print(f"{asin}\tR:{rating}")

        except (ValueError, IndexError):
            continue

if __name__ == '__main__':
    mapper()

Overwriting mapper_task4.py


In [ ]:
%%writefile reduce_task4.py
import sys

def reducer():
    current_asin = None
    rating = None
    keywords = []

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        asin, data = line.split('\t', 1)

        if current_asin != asin:
            # Output
            if current_asin and rating and keywords:
                # sorted the most frequency word
                sorted_keywords = sorted(keywords, key=lambda x: x[1], reverse=True)
                keywords_str = ", ".join([f"{word}({count})" for word, count in sorted_keywords])
                print(f"{current_asin}\t{rating}\t{keywords_str}")

            # update current_asin
            current_asin = asin
            rating = None
            keywords = []

        # Rating data
        if data.startswith('R:'):
            rating = data[2:]
        # Keyword data
        elif data.startswith('K:'):
            keyword_data = data[2:]
            if ':' in keyword_data:
                word, count = keyword_data.split(':', 1)
                keywords.append((word, int(count)))

    # Output for  the last line
    if current_asin and rating and keywords:
        sorted_keywords = sorted(keywords, key=lambda x: x[1], reverse=True)
        keywords_str = ", ".join([f"{word}({count})" for word, count in sorted_keywords])
        print(f"{current_asin}\t{rating}\t{keywords_str}")

if __name__ == "__main__":
    reducer()

Writing reduce_task4.py


In [ ]:
print("Running MapReduce job for Task 4")

# find Tasks 2 et 3
!cat task2_output.txt task3_output.txt | python mapper_task4.py | sort | python reduce_task4.py > task4_output.txt

print("\nFirst 10 lines of task4_output.txt:")
!head -10 task4_output.txt

Running MapReduce job for Task 4

First 10 lines of task4_output.txt:
0005019281	4.42	christmas(166), winkler(126), movie(123), henry(106), carol(96), great(90), good(79), story(75), version(75), classic(69), scrooge(59), one(49), american(43), dickens(40), well(39), slade(34), time(33), love(32), like(31), old(30), original(30), twist(29), watch(26), best(24), depression(24), man(24), character(23), dvd(23), excellent(23), acting(22), job(22), movies(22), nice(22), really(22), set(21), would(20), favorite(19), new(19), america(18), made(18), see(18), still(18), tv(18), family(17), versions(17), always(16), different(16), enjoyed(16), fonz(16), loved(16), actor(15), film(15), much(15), better(14), collection(14), first(14), future(14), seen(14), watched(14), years(14), make(13), people(13), role(13), wonderful(13), england(12), every(12), happy(12), holiday(12), part(12), past(12), take(12), young(12), adaptation(11), came(11), charles(11), even(11), liked(11), saw(11), ago(10), also(1